# 1.混合检索示例

知识点讲解：EnsembleRetriever 集成多种检索算法实现混合检索

1. 稀疏检索与稠密检索的互补关系
   - 稀疏检索（BM25 为代表）：基于词频统计，向量维度=词表大小且极稀疏；强项是精确术语匹配（产品型号、错误码、人名、API 名）；弱项是不懂语义
   - 稠密检索（Embedding 向量检索）：基于神经网络语义表示，向量维度固定（如 1536）且稠密；强项是理解语义与同义改写；弱项是对罕见专有名词泛化不足
   - 两者失效场景几乎正交，混合使用可显著提升整体召回率，即业界公认的 Hybrid Search

2. BM25 算法要点
   - TF-IDF 的改进版，传统信息检索事实标准
   - 词频饱和：某词出现 10 次与 100 次得分差距有限，避免关键词堆砌
   - 文档长度归一化：惩罚长文档以保证公平
   - BM25Retriever 纯本地内存计算，零 API 成本、毫秒级延迟

3. EnsembleRetriever 的融合机制（RRF）
   - 接受检索器列表 retrievers 与权重 weights，并行调用各子检索器的 get_relevant_documents() 得到有序结果
   - 用带权 RRF 融合：score(d) = Σ weight_i * 1 / (k + rank_i(d))，k 默认 60
   - 为何用 RRF 而非直接加权得分：BM25 得分域 [0,+∞)、余弦相似度 [-1,1]，量纲不同无法直接相加；RRF 只用"排名"天然解决跨系统量纲问题
   - 多路都出现的文档累加得分，形成"多路共识"加权

4. weights 权重调优
   - 列表长度必须与 retrievers 一致、一一对应，内部自动归一化
   - 技术文档 / 代码库 / 含大量专有名词 → 提高 BM25 权重（如 [0.6, 0.4]）
   - 口语化提问 / 客服对话 / 语义泛化强 → 提高向量权重（如 [0.3, 0.7]）
   - 无先验从 [0.5, 0.5] 起步，按评测集调整

5. 本例的经典示例设计
   - 查询："除了猫，你养了什么宠物呢？"
   - BM25 路：精确命中含"猫"文档（page=1、3），但无法理解"宠物"上位概念，命中不了"狗"
   - 向量路：理解"宠物"语义，召回"狗"（page=10）与"猫"相关文档
   - 融合后：字面精确 + 语义泛化，完整覆盖用户真实意图，直观展示 1+1>2

6. EnsembleRetriever 的扩展能力与生态
   - retrievers 可放任意数量、任意类型 BaseRetriever，也能混入自定义检索器、MultiQueryRetriever、HyDERetriever、ES / 图数据库检索器
   - 混合检索被 Dify、Coze、智谱 等 AI 开发平台广泛采用，也是本课程 LLMOps 项目使用的检索策略
   - 运行时还可通过 configurable_fields 动态控制某路检索的 k 值与权重

7. 生产实践：多路召回 → RRF 融合 → ReRank 精排
   - 这是构建生产级 RAG 召回层的标准做法
   - 各路 k 值建议设为最终需要条数的 2~3 倍，给 RRF 留足融合空间
   - 混合检索之后再接 Cross-Encoder ReRank 可进一步提升精排质量

最佳实践建议

- 生产级 RAG 的召回层强烈建议使用混合检索，纯向量检索在专有名词场景漏召回严重
- weights 必须基于真实业务评测集调优，不要凭直觉拍定
- 各路 k 值建议设为最终需要条数的 2~3 倍，给 RRF 留足融合空间
- BM25 的中文分词需要特别注意：BM25Retriever 默认按空格切分，对中文几乎等同于整句匹配；生产环境应传入 jieba 等中文分词器作为 preprocess_func，否则 BM25 路效果大打折扣
- BM25Retriever 数据常驻内存，不适合超大规模语料；大数据量应改用 Elasticsearch / OpenSearch 的 BM25 能力
- 两路检索器的数据源应保持一致（同一批 documents），否则融合结果会不完整
- 混合检索之后再接 Cross-Encoder ReRank（第 52 节）可进一步提升精排质量，形成"多路召回 → RRF 融合 → 精排"的标准三段式架构
- EnsembleRetriever 内部是顺序调用子检索器的，若某路较慢会拖累整体延迟，高性能场景可自行实现基于 abatch 的并发版本


In [ ]:
import dotenv
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
dotenv.load_dotenv()
documents = [
    Document(page_content="笨笨是一只很喜欢睡觉的猫咪", metadata={"page": 1}),
    Document(page_content="我喜欢在夜晚听音乐，这让我感到放松。", metadata={"page": 2}),
    Document(page_content="猫咪在窗台上打盹，看起来非常可爱。", metadata={"page": 3}),
    Document(page_content="学习新技能是每个人都应该追求的目标。", metadata={"page": 4}),
    Document(page_content="我最喜欢的食物是意大利面，尤其是番茄酱的那种。", metadata={"page": 5}),
    Document(page_content="昨晚我做了一个奇怪的梦，梦见自己在太空飞行。", metadata={"page": 6}),
    Document(page_content="我的手机突然关机了，让我有些焦虑。", metadata={"page": 7}),
    Document(page_content="阅读是我每天都会做的事情，我觉得很充实。", metadata={"page": 8}),
    Document(page_content="他们一起计划了一次周末的野餐，希望天气能好。", metadata={"page": 9}),
    Document(page_content="我的狗喜欢追逐球，看起来非常开心。", metadata={"page": 10}),
]
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 4
faiss_db = FAISS.from_documents(documents, embedding=OpenAIEmbeddings(model="text-embedding-3-small"))
faiss_retriever = faiss_db.as_retriever(search_kwargs={"k": 4})
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5],
)
docs = ensemble_retriever.invoke("除了猫，你养了什么宠物呢？")
print(docs)
print(len(docs))